# ECG (PTB-XL) — train from scratch

**Vertical 3 from `docs/12-training-plan.md`.** This is the one component in the plan trained **from scratch**, because no credible pretrained 12-lead ECG foundation model exists on Hugging Face today (re-check before you run this — that may have changed).

**What this notebook does:**
1. Loads PTB-XL's official metadata and **official patient-disjoint stratified folds** (never re-split these — that's how PTB-XL benchmark numbers stay comparable).
2. Maps PTB-XL's SCP-ECG statement codes to the 5 diagnostic superclasses (`NORM`, `MI`, `STTC`, `CD`, `HYP`).
3. Trains a 1D-CNN (ResNet-style) multi-label classifier.
4. Evaluates with **macro-AUROC** (the PTB-XL-standard metric) and tracks **MI sensitivity separately** — a missed MI is a different risk tier than a missed minor conduction abnormality, so it doesn't get averaged away.
5. Saves a `.pt` checkpoint you can wire into a `SpecialistPort` adapter the same way `ModelBackedChestXRaySpecialistAdapter` wraps an HTTP endpoint.

## Kaggle setup (do this before running)
- **Add data:** click *Add Data* → search **"PTB-XL"** → add a PTB-XL mirror dataset (several exist on Kaggle; any that includes `ptbxl_database.csv`, `scp_statements.csv`, and the per-record `.dat`/`.hea` (WFDB) files works — this notebook auto-detects the path).
- **Accelerator:** Settings → Accelerator → **GPU T4 x2** (or any GPU). This is the cheapest vertical in the plan — a few hours on a single GPU is enough.
- **Internet:** on (needed for `pip install wfdb neurokit2`).

> If you'd rather work from the canonical source instead of a Kaggle mirror, the official release is [PhysioNet PTB-XL](https://physionet.org/content/ptb-xl/) — download it and point `PTBXL_ROOT` below at the extracted folder.

In [ ]:
# Cell 1 — install the two packages Kaggle's base image doesn't ship with.
# wfdb reads the raw PTB-XL waveform format; neurokit2 is used for optional
# R-peak/beat-segmentation features (docs/12 calls out this exact pairing).
!pip install -q wfdb neurokit2

In [ ]:
# Cell 2 — imports and reproducibility.
# Training principle 3 in docs/12: fixed, versioned splits, reproducible runs.
import glob
import json
import os
import random

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import wfdb
from sklearn.metrics import roc_auc_score
from torch.utils.data import DataLoader, Dataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

## Locate the dataset

Different Kaggle PTB-XL mirrors unpack to slightly different paths. We search
`/kaggle/input` for `ptbxl_database.csv` rather than hardcoding a dataset slug,
so this cell works regardless of which mirror you added.

In [ ]:
# Cell 3 — auto-detect the PTB-XL root directory.
candidates = glob.glob("/kaggle/input/**/ptbxl_database.csv", recursive=True)
if not candidates:
    raise FileNotFoundError(
        "Could not find ptbxl_database.csv under /kaggle/input. "
        "Add a PTB-XL dataset via 'Add Data', or set PTBXL_ROOT manually below."
    )
PTBXL_ROOT = os.path.dirname(candidates[0])
print("PTB-XL root:", PTBXL_ROOT)

## Build multi-label superclass targets

PTB-XL's `scp_codes` column holds a dict of SCP-ECG statement codes → confidence
(0–100). `scp_statements.csv` maps each code to one of 5 diagnostic superclasses.
A record can belong to more than one superclass (multi-label), matching the
`Finding[]`-shaped multi-label task described in docs/12.

In [ ]:
# Cell 4 — load metadata and the SCP statement → superclass mapping.
SUPERCLASSES = ["NORM", "MI", "STTC", "CD", "HYP"]

ptbxl_df = pd.read_csv(os.path.join(PTBXL_ROOT, "ptbxl_database.csv"), index_col="ecg_id")
ptbxl_df.scp_codes = ptbxl_df.scp_codes.apply(lambda x: eval(x))  # stored as a string-repr dict

scp_df = pd.read_csv(os.path.join(PTBXL_ROOT, "scp_statements.csv"), index_col=0)
scp_df = scp_df[scp_df.diagnostic == 1]  # only diagnostic statements map to a superclass


def scp_codes_to_superclasses(scp_codes: dict) -> list[str]:
    labels = set()
    for code in scp_codes.keys():
        if code in scp_df.index:
            labels.add(scp_df.loc[code].diagnostic_class)
    return sorted(labels & set(SUPERCLASSES))


ptbxl_df["superclasses"] = ptbxl_df.scp_codes.apply(scp_codes_to_superclasses)
ptbxl_df = ptbxl_df[ptbxl_df.superclasses.apply(len) > 0]  # drop records with no mapped diagnosis

for superclass in SUPERCLASSES:
    ptbxl_df[superclass] = ptbxl_df.superclasses.apply(lambda labels: int(superclass in labels))

print(ptbxl_df[SUPERCLASSES].sum())

## Use the official folds — do not re-split

`strat_fold` (1–10) is PTB-XL's own patient-disjoint stratified fold assignment.
The dataset's authors recommend: **fold 10 = test, fold 9 = validation, folds
1–8 = train.** Using these exactly is what makes your macro-AUROC comparable to
published PTB-XL benchmarks (docs/12, Vertical 3: "use them as-is").

In [ ]:
# Cell 5 — official train/val/test split by strat_fold.
train_df = ptbxl_df[ptbxl_df.strat_fold <= 8]
val_df = ptbxl_df[ptbxl_df.strat_fold == 9]
test_df = ptbxl_df[ptbxl_df.strat_fold == 10]
print(f"train={len(train_df)} val={len(val_df)} test={len(test_df)}")

## Dataset & preprocessing

We use the **100Hz** release (`filename_lr`) rather than 500Hz — it's one of
PTB-XL's two shipped sampling rates and keeps training fast on a single Kaggle
GPU without materially hurting the 5-superclass classification task. Each
12-lead signal is z-normalized **per lead** (docs/12: "per-lead z-normalization").

In [ ]:
# Cell 6 — Dataset class: reads a WFDB record, resamples is a no-op here since
# filename_lr is already 100Hz, and per-lead normalizes.
class PTBXLDataset(Dataset):
    def __init__(self, dataframe: pd.DataFrame, root: str):
        self.dataframe = dataframe.reset_index()
        self.root = root

    def __len__(self) -> int:
        return len(self.dataframe)

    def __getitem__(self, index: int):
        row = self.dataframe.iloc[index]
        record_path = os.path.join(self.root, row.filename_lr)
        signal, _ = wfdb.rdsamp(record_path)  # shape: (1000 samples, 12 leads) at 100Hz
        signal = signal.T.astype("float32")  # -> (12, 1000)

        mean = signal.mean(axis=1, keepdims=True)
        std = signal.std(axis=1, keepdims=True) + 1e-6
        signal = (signal - mean) / std

        labels = row[SUPERCLASSES].values.astype("float32")
        return torch.from_numpy(signal), torch.from_numpy(labels)


train_ds = PTBXLDataset(train_df, PTBXL_ROOT)
val_ds = PTBXLDataset(val_df, PTBXL_ROOT)
test_ds = PTBXLDataset(test_df, PTBXL_ROOT)

train_loader = DataLoader(train_ds, batch_size=128, shuffle=True, num_workers=2, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=256, shuffle=False, num_workers=2)
test_loader = DataLoader(test_ds, batch_size=256, shuffle=False, num_workers=2)

## Model — 1D-CNN (ResNet-style)

docs/12: *"train a 1D-CNN or CNN+transformer hybrid directly on PTB-XL."* We use
a compact 1D ResNet: strided conv blocks for downsampling, residual connections
for trainability, global average pooling, then a linear head over the 5
superclasses. Small enough to train from scratch in a few hours on one GPU.

In [ ]:
# Cell 7 — model definition.
class ResidualBlock1D(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, stride: int = 1):
        super().__init__()
        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size=7, stride=stride, padding=3, bias=False)
        self.bn1 = nn.BatchNorm1d(out_channels)
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size=7, padding=3, bias=False)
        self.bn2 = nn.BatchNorm1d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.downsample = None
        if stride != 1 or in_channels != out_channels:
            self.downsample = nn.Sequential(
                nn.Conv1d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm1d(out_channels),
            )

    def forward(self, x):
        identity = x if self.downsample is None else self.downsample(x)
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return self.relu(out + identity)


class ECGResNet1D(nn.Module):
    def __init__(self, in_leads: int = 12, num_classes: int = len(SUPERCLASSES)):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv1d(in_leads, 32, kernel_size=15, stride=2, padding=7, bias=False),
            nn.BatchNorm1d(32),
            nn.ReLU(inplace=True),
        )
        self.layer1 = ResidualBlock1D(32, 64, stride=2)
        self.layer2 = ResidualBlock1D(64, 128, stride=2)
        self.layer3 = ResidualBlock1D(128, 256, stride=2)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.head = nn.Linear(256, num_classes)

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.pool(x).squeeze(-1)
        return self.head(x)


model = ECGResNet1D().to(DEVICE)
print(sum(p.numel() for p in model.parameters()) / 1e6, "M parameters")

## Loss — weighted multi-label BCE

docs/12: *"weighted multi-label BCE (SCP-ECG statements are multi-label and
imbalanced — MI-related labels are rarer than 'normal')."* We compute per-class
positive weights from the training set's class frequencies and pass them to
`BCEWithLogitsLoss`.

In [ ]:
# Cell 8 — class-imbalance-aware loss.
positive_counts = train_df[SUPERCLASSES].sum().values
negative_counts = len(train_df) - positive_counts
pos_weight = torch.tensor(negative_counts / np.clip(positive_counts, 1, None), dtype=torch.float32).to(DEVICE)
print("pos_weight per class:", dict(zip(SUPERCLASSES, pos_weight.tolist())))

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)

## Training loop

Standard supervised loop with validation-set macro-AUROC tracked each epoch,
keeping the best checkpoint — this *is* the promotion-gate check from
docs/12/docs/10 ("beat a trivial baseline", "no subgroup regression") in miniature;
the full gate (subgroup breakdown, registry entry) happens outside this notebook
before the checkpoint is ever wired into a `SpecialistPort` adapter.

In [ ]:
# Cell 9 — training + validation loop.
def macro_auroc(y_true: np.ndarray, y_score: np.ndarray) -> float:
    scores = []
    for class_index in range(y_true.shape[1]):
        if len(np.unique(y_true[:, class_index])) < 2:
            continue  # AUROC is undefined for a class with only one label value present
        scores.append(roc_auc_score(y_true[:, class_index], y_score[:, class_index]))
    return float(np.mean(scores))


@torch.no_grad()
def evaluate(loader: DataLoader) -> tuple[float, np.ndarray, np.ndarray]:
    model.eval()
    all_true, all_score = [], []
    for signals, labels in loader:
        signals = signals.to(DEVICE)
        logits = model(signals)
        all_true.append(labels.numpy())
        all_score.append(torch.sigmoid(logits).cpu().numpy())
    y_true = np.concatenate(all_true)
    y_score = np.concatenate(all_score)
    return macro_auroc(y_true, y_score), y_true, y_score


NUM_EPOCHS = 20
best_val_auroc = 0.0
best_state_dict = None

for epoch in range(NUM_EPOCHS):
    model.train()
    running_loss = 0.0
    for signals, labels in train_loader:
        signals, labels = signals.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        logits = model(signals)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * signals.size(0)
    scheduler.step()

    train_loss = running_loss / len(train_ds)
    val_auroc, _, _ = evaluate(val_loader)
    print(f"epoch {epoch+1:02d}  train_loss={train_loss:.4f}  val_macro_auroc={val_auroc:.4f}")

    if val_auroc > best_val_auroc:
        best_val_auroc = val_auroc
        best_state_dict = {k: v.cpu().clone() for k, v in model.state_dict().items()}

model.load_state_dict(best_state_dict)
print("Best val macro-AUROC:", best_val_auroc)

## Final evaluation — held-out test fold

Reports macro-AUROC (the PTB-XL-standard metric) **and** MI sensitivity at a
0.5 threshold separately, per docs/12: *"track it separately from the aggregate
score"* — a missed MI is a different risk tier than a missed minor finding.

In [ ]:
# Cell 10 — held-out test evaluation.
test_auroc, y_true, y_score = evaluate(test_loader)
print("Test macro-AUROC:", test_auroc)

for class_index, superclass in enumerate(SUPERCLASSES):
    class_auroc = roc_auc_score(y_true[:, class_index], y_score[:, class_index])
    print(f"  {superclass}: AUROC={class_auroc:.4f}")

mi_index = SUPERCLASSES.index("MI")
mi_predictions = (y_score[:, mi_index] >= 0.5).astype(int)
mi_true_positive = ((mi_predictions == 1) & (y_true[:, mi_index] == 1)).sum()
mi_actual_positive = (y_true[:, mi_index] == 1).sum()
mi_sensitivity = mi_true_positive / max(mi_actual_positive, 1)
print(f"MI sensitivity @0.5 threshold: {mi_sensitivity:.4f}  ({mi_true_positive}/{mi_actual_positive})")

## Save the checkpoint

Saves a plain `state_dict` `.pt` file — portable, loadable without this notebook's
exact class definitions being importable elsewhere as long as you re-declare
`ECGResNet1D`. Per docs/03/docs/04: **track this checkpoint with DVC, never commit
it to git**, and register it in the model registry (docs/10) in `shadow` status
before any real traffic reaches it.

In [ ]:
# Cell 11 — save checkpoint + a small metadata sidecar for the registry.
checkpoint_path = "/kaggle/working/ecg_ptbxl_resnet1d.pt"
torch.save(model.state_dict(), checkpoint_path)

metadata = {
    "model": "ECGResNet1D",
    "superclasses": SUPERCLASSES,
    "sampling_rate_hz": 100,
    "input_shape": [12, 1000],
    "test_macro_auroc": test_auroc,
    "mi_sensitivity_at_0.5": float(mi_sensitivity),
    "trained_on": "PTB-XL, official strat_fold split (train 1-8, val 9, test 10)",
}
with open("/kaggle/working/ecg_ptbxl_resnet1d.json", "w") as handle:
    json.dump(metadata, handle, indent=2)

print("Saved:", checkpoint_path)
print(json.dumps(metadata, indent=2))

## Next steps (outside this notebook)

1. Download `ecg_ptbxl_resnet1d.pt` from Kaggle's *Output* tab.
2. Run the **subgroup breakdown** (age, sex, and site if available) before promoting — docs/06/docs/10 require this, and it isn't in this notebook.
3. Fit **post-hoc calibration** (temperature scaling) on a held-out calibration split — never trust the raw sigmoid as a probability (docs/12, principle 6).
4. Serve it behind an HTTP endpoint (e.g., a small FastAPI wrapper on Cloud Run/Vertex AI — see the earlier discussion on hosting), then point a new ECG `SpecialistPort` adapter's `endpoint_url` at it, following the exact pattern `ModelBackedChestXRaySpecialistAdapter` already uses.